In [ ]:
from google.colab import files
uploaded=files.upload()
a=next(iter(uploaded))

Saving recruitmentdataset-2022-1.3.csv to recruitmentdataset-2022-1.3.csv


In [ ]:
from google.colab import files
uploaded=files.upload()
b=next(iter(uploaded))

Saving 2025-fairness-recruitment-dataset.csv to 2025-fairness-recruitment-dataset.csv


In [ ]:
import numpy as np
import pandas as pd
import sklearn
import matplotlib.pyplot as plt

In [ ]:
df1=pd.read_csv(a)
#df2=pd.read_csv(b)

df1.head(10)

,Id,gender,age,nationality,sport,ind-university_grade,ind-debateclub,ind-programming_exp,ind-international_exp,ind-entrepeneur_exp,ind-languages,ind-exact_study,ind-degree,company,decision
0,x8011e,female,24,German,Swimming,70,False,False,False,False,1,True,phd,A,True
1,x6077a,male,26,German,Golf,67,False,True,False,False,2,True,bachelor,A,False
2,x6006e,female,23,Dutch,Running,67,False,True,True,False,0,True,master,A,False
3,x2173b,male,24,Dutch,Cricket,70,False,True,False,False,1,True,master,A,True
4,x6241a,female,26,German,Golf,59,False,False,False,False,1,False,master,A,True
5,x9063d,female,26,Dutch,Chess,63,False,False,False,False,1,True,bachelor,A,True
6,x5785d,female,27,Dutch,Tennis,63,True,True,False,False,2,True,bachelor,A,False
7,x8767c,female,22,Dutch,Swimming,71,False,True,False,False,1,True,master,A,True
8,x6541b,female,28,Dutch,Football,65,True,False,False,True,3,False,bachelor,A,False
9,x3890b,male,24,Dutch,Football,55,True,False,False,True,3,False,master,A,True


In [ ]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Id                     4000 non-null   object
 1   gender                 4000 non-null   object
 2   age                    4000 non-null   int64 
 3   nationality            4000 non-null   object
 4   sport                  4000 non-null   object
 5   ind-university_grade   4000 non-null   int64 
 6   ind-debateclub         4000 non-null   bool  
 7   ind-programming_exp    4000 non-null   bool  
 8   ind-international_exp  4000 non-null   bool  
 9   ind-entrepeneur_exp    4000 non-null   bool  
 10  ind-languages          4000 non-null   int64 
 11  ind-exact_study        4000 non-null   bool  
 12  ind-degree             4000 non-null   object
 13  company                4000 non-null   object
 14  decision               4000 non-null   bool  
dtypes: bool(6), int64(3),

In [ ]:
df1.isnull().sum()

,0
Id,0
gender,0
age,0
nationality,0
sport,0
ind-university_grade,0
ind-debateclub,0
ind-programming_exp,0
ind-international_exp,0
ind-entrepeneur_exp,0


In [ ]:
df1.duplicated().sum()

np.int64(0)

In [ ]:
df1 =df1.drop('Id',axis=1)

In [ ]:
df1['age_group'] = pd.cut(
df1['age'],
bins=[0, 25, 40, 60, 100],
labels=['young', 'adult', 'middle', 'senior']
)


In [ ]:
df1.drop(columns=['age'], inplace=True)

In [ ]:
X_train = df1.drop(columns=['decision'])
y_train = df1['decision']

from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(
    df1,
    test_size=0.2,
    random_state=42,
    stratify=df1['decision']
)


In [ ]:
def entropy(y):
    values, counts = np.unique(y, return_counts=True)
    probabilities = counts / counts.sum()
    return -np.sum(probabilities * np.log2(probabilities + 1e-9))

def information_gain(data, feature, target):
    total_entropy = entropy(data[target])

    values, counts = np.unique(data[feature], return_counts=True)
    weighted_entropy = 0

    for v, count in zip(values, counts):
        subset = data[data[feature] == v]
        weighted_entropy += (count / len(data)) * entropy(subset[target])

    return total_entropy - weighted_entropy

def root_node(data, features, target):
    gains = []
    for feature in features:
        gains.append(information_gain(data, feature, target))

    return features[np.argmax(gains)]



In [ ]:
def id3(data, features, target):

    if len(data[target].unique()) == 1:
        return data[target].iloc[0]

    if not features:
        return data[target].mode()[0]

    best = root_node(data, features, target)
    tree = {best: {}}

    remaining = [f for f in features if f != best]

    for value in data[best].unique():
        subset = data[data[best] == value]
        tree[best][value] = (
            data[target].mode()[0]
            if len(subset) == 0
            else id3(subset, remaining, target)
        )

    return tree


In [ ]:
def predict_one(sample, tree):

    if not isinstance(tree, dict):
        return tree

    feature = list(tree.keys())[0]
    value = sample[feature]

    if value in tree[feature]:
        return predict_one(sample, tree[feature][value])
    else:
        return None


def predict(data, tree):
    return data.apply(lambda row: predict_one(row, tree), axis=1)


In [ ]:
target = 'decision'
features = [col for col in train_data.columns if col != target]

tree = id3(train_data, features, target)
predictions = predict(test_data, tree)





In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_true = test_data['decision']

default_prediction_value = train_data['decision'].mode()[0]

y_pred = predictions.fillna(default_prediction_value)

print("Accuracy:", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred))
print("Recall:", recall_score(y_true, y_pred))
print("F1-score:", f1_score(y_true, y_pred))


Accuracy: 0.8025
Precision: 0.7513227513227513
Recall: 0.5612648221343873
F1-score: 0.6425339366515838


/tmp/ipython-input-274/1698188581.py:7: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y_pred = predictions.fillna(default_prediction_value)


In [ ]:
test_data = test_data.copy()
test_data['prediction'] = predictions


In [ ]:
test_data.groupby('gender')['prediction'].mean()


,prediction
gender,
female,0.230284
male,0.313559
other,0.181818


In [ ]:
print(test_data.groupby('age_group')['prediction'].mean())


age_group
young     0.259398
adult     0.286058
middle         NaN
senior         NaN
Name: prediction, dtype: object


/tmp/ipython-input-274/130643619.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(test_data.groupby('age_group')['prediction'].mean())
